In [5]:
import pypdf
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────────────
PDF_PATH = Path("../data/NPPF_December_2024.pdf")
CHUNK_SIZE = 500        # characters per chunk
CHUNK_OVERLAP = 50      # characters of overlap between consecutive chunks

# ── Step 1: Load the PDF and extract raw text page by page ──────────────────
reader = pypdf.PdfReader(PDF_PATH)
print(f"Total pages: {len(reader.pages)}")

pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:                          # some pages are blank or image-only
        pages.append({"page": i + 1, "text": text.strip()})

print(f"Pages with extractable text: {len(pages)}")
print(f"\n--- Sample: page 1 (first 300 chars) ---")
print(pages[1]["text"][:300])

Total pages: 82
Pages with extractable text: 82

--- Sample: page 1 (first 300 chars) ---
© Crown copyright 2024 
 
Copyright in the typographical arrangement rests with the Crown. 
 
You may re-use this information (not including logos) free of charge in any format or 
medium, under the terms of the Open Government Licence. To view this licence visit 
http://www.nationalarchives.gov.uk/


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Config ───────────────────────────────────────────────────────────────────
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# ── Step 2: Concatenate all page text, keeping page metadata ─────────────────
# We'll chunk per-page so we can attach page number metadata to each chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]  # tries these in order
)

chunks = []
for page in pages:
    page_chunks = splitter.split_text(page["text"])
    for i, chunk_text in enumerate(page_chunks):
        chunks.append({
            "chunk_id": f"p{page['page']}_c{i}",
            "page": page["page"],
            "text": chunk_text
        })

print(f"Total chunks: {len(chunks)}")
print(f"Avg chunk length: {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")
print(f"\n--- Sample chunk ---")
print(f"ID: {chunks[10]['chunk_id']}")
print(f"Page: {chunks[10]['page']}")
print(f"Text: {chunks[10]['text']}")

Total chunks: 493
Avg chunk length: 424 chars

--- Sample chunk ---
ID: p4_c4
Page: 4
Text: policy statements form part of the overall framework of national planning policy, and 
may be a material consideration in preparing plans and making decisions on 
planning applications. 
 
6. Other statements of government policy may be material when preparing plans or 
deciding applications, such as relevant Written Ministerial Statements and endorsed 
recommendations of the National Infrastructure Commission.
